# LCT — Target-domain fine-tuning impact

Сравнение **одной и той же архитектуры RF-DETR Medium @640**:

### BEFORE TARGET FT
External RF-DETR Medium @640 после обучения только на `construction_v1`.

### AFTER TARGET FT
FINAL80 RF-DETR Medium @640 после fine-tuning на всех 80 размеченных изображениях организаторов.

Сравнение проводится на **одних и тех же 20 TEST изображениях**.

Цель: показать эффект target-domain adaptation отдельно от смены архитектуры и resolution.

Фиксировано:
- architecture: RF-DETR Medium
- resolution: 640
- full-frame inference
- operating threshold: 0.25
- IoU for TP/FP/FN: 0.50
- TEST не используется для tuning

Notebook считает:
- mAP50-95 / mAP50 / mAP75
- Precision / Recall / F1
- TP / FP / FN
- per-class AP
- recall по относительному размеру объекта
- per-image изменения
- 20 визуальных сравнений `GT | BEFORE TARGET FT | AFTER FINAL80`
- summary plots для README / презентации


In [ ]:
!pip install -q "rfdetr[train]==1.10.1" pycocotools pyyaml pandas pillow matplotlib

from pathlib import Path
import io, json, math, time, shutil, zipfile
from contextlib import redirect_stdout

import numpy as np
import pandas as pd
import torch
import yaml
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt

from rfdetr import RFDETRMedium
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

RESULTS = WORK / "target_ft_impact"
VIS = RESULTS / "visuals"
RESULTS.mkdir(parents=True, exist_ok=True)
VIS.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = [
    "excavator",
    "dump_truck",
    "truck",
    "loader",
    "bulldozer",
    "motor_grader",
    "roller",
    "concrete_mixer",
    "telehandler",
    "piling_machine",
    "crane_manipulator",
    "mobile_crane",
    "tower_crane",
]
NAME_TO_ID = {n:i for i,n in enumerate(CLASS_NAMES)}

AP_CONF = 0.001
OP_CONF = 0.25
MATCH_IOU = 0.50

# Set exact paths here only if auto-discovery is ambiguous.
MANUAL_BEFORE_CKPT = None
MANUAL_AFTER_CKPT = None

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# ============================================================
# Locate organizer split.
# ============================================================

def normalize_names(raw):
    if isinstance(raw, dict):
        return [raw[i] if i in raw else raw[str(i)] for i in range(len(raw))]
    return list(raw)

candidates = []
for y in INPUT.rglob("data.yaml"):
    root = y.parent
    if (
        (root / "train" / "images").exists()
        and (root / "val" / "images").exists()
        and (root / "test" / "images").exists()
        and (root / "test" / "labels").exists()
    ):
        candidates.append(y)

preferred = [
    y for y in candidates
    if "lct-organizer100-split-v1" in y.as_posix().lower()
    or "organizer100_split" in y.as_posix().lower()
]

assert preferred or candidates, "Attach grigoriyyy/lct-organizer100-split-v1"

DATA_YAML = (preferred or candidates)[0]
DATA_ROOT = DATA_YAML.parent

with open(DATA_YAML, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

assert normalize_names(cfg["names"]) == CLASS_NAMES, "Taxonomy mismatch"

EXTS = {".jpg",".jpeg",".png",".webp",".bmp",".tif",".tiff"}
TEST_IMAGES = sorted(
    p for p in (DATA_ROOT / "test" / "images").iterdir()
    if p.is_file() and p.suffix.lower() in EXTS
)

assert len(TEST_IMAGES) == 20, f"Expected 20 TEST images, got {len(TEST_IMAGES)}"

print("DATA_ROOT:", DATA_ROOT)
print("TEST:", len(TEST_IMAGES))


In [ ]:
# ============================================================
# Locate BEFORE TARGET FT:
# external RF-DETR Medium@640 checkpoint.
#
# Correct source:
# output of lct-rfdetr-medium-640-finetune.ipynb
# ============================================================

if MANUAL_BEFORE_CKPT:
    BEFORE_CKPT = Path(MANUAL_BEFORE_CKPT)
    assert BEFORE_CKPT.exists()
else:
    forbidden = [
        "target_ft", "target-ft", "organizer",
        "final80", "continue", "continuation",
        "crop", "branch_aware",
    ]

    candidates = []
    for fn in [
        "checkpoint_best_ema.pth",
        "checkpoint_best_total.pth",
        "checkpoint_best_regular.pth",
    ]:
        for p in INPUT.rglob(fn):
            low = p.as_posix().lower()
            if "medium" not in low or "640" not in low:
                continue
            if any(x in low for x in forbidden):
                continue
            candidates.append((0 if fn=="checkpoint_best_ema.pth" else 1, len(p.as_posix()), p))

    assert candidates, (
        "External Medium@640 checkpoint not found. "
        "Attach output of lct-rfdetr-medium-640-finetune.ipynb."
    )
    BEFORE_CKPT = sorted(candidates)[0][2]

print("BEFORE TARGET FT:")
print(BEFORE_CKPT)


In [ ]:
# ============================================================
# Locate AFTER TARGET FT:
# FINAL80 final_model.pth.
# ============================================================

if MANUAL_AFTER_CKPT:
    AFTER_CKPT = Path(MANUAL_AFTER_CKPT)
    assert AFTER_CKPT.exists()
else:
    hits = [
        p for p in INPUT.rglob("final_model.pth")
        if "final80" in p.as_posix().lower()
        or "deployment" in p.as_posix().lower()
    ]

    if hits:
        AFTER_CKPT = sorted(hits, key=lambda p: len(p.as_posix()))[0]
    else:
        zips = list(INPUT.rglob("*final80*deployment*.zip"))
        assert zips, (
            "FINAL80 checkpoint not found. Attach FINAL80 output "
            "containing final_model.pth or final80_deployment_bundle.zip."
        )
        ex = WORK / "_final80_extract"
        if ex.exists():
            shutil.rmtree(ex)
        ex.mkdir()
        with zipfile.ZipFile(zips[0], "r") as z:
            z.extractall(ex)
        extracted = list(ex.rglob("final_model.pth"))
        assert extracted, "final_model.pth not found inside FINAL80 bundle"
        AFTER_CKPT = extracted[0]

print("\nAFTER TARGET FT (FINAL80):")
print(AFTER_CKPT)

assert BEFORE_CKPT.resolve() != AFTER_CKPT.resolve()


In [ ]:
# ============================================================
# Build COCO GT from 20 TEST labels.
# ============================================================

images = []
annotations = []
gt_rows = []
ann_id = 1

for image_id, img_path in enumerate(TEST_IMAGES, start=1):
    with Image.open(img_path) as im:
        w, h = im.size

    images.append({
        "id": image_id,
        "file_name": img_path.name,
        "width": w,
        "height": h,
    })

    label_path = DATA_ROOT / "test" / "labels" / f"{img_path.stem}.txt"
    assert label_path.exists(), label_path

    for line in label_path.read_text(encoding="utf-8").splitlines():
        if not line.strip():
            continue

        parts = line.split()
        cls = int(float(parts[0]))
        xc, yc, bw, bh = map(float, parts[1:5])

        x = max(0.0, (xc - bw / 2) * w)
        y = max(0.0, (yc - bh / 2) * h)
        ww = min(bw * w, w - x)
        hh = min(bh * h, h - y)

        if ww <= 0 or hh <= 0:
            continue

        area = ww * hh

        annotations.append({
            "id": ann_id,
            "image_id": image_id,
            "category_id": cls + 1,
            "bbox": [x, y, ww, hh],
            "area": area,
            "iscrowd": 0,
        })

        gt_rows.append({
            "ann_id": ann_id,
            "image_id": image_id,
            "image_name": img_path.name,
            "class_id": cls,
            "class_name": CLASS_NAMES[cls],
            "x1": x,
            "y1": y,
            "x2": x + ww,
            "y2": y + hh,
            "relative_area": area / (w * h),
        })

        ann_id += 1

payload = {
    "info": {},
    "licenses": [],
    "images": images,
    "annotations": annotations,
    "categories": [
        {"id": i+1, "name": n, "supercategory": "equipment"}
        for i, n in enumerate(CLASS_NAMES)
    ],
}

GT_JSON = RESULTS / "test_gt.json"
GT_JSON.write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")

GT = pd.DataFrame(gt_rows)
GT.to_csv(RESULTS / "test_gt_boxes.csv", index=False)

COCO_GT = COCO(str(GT_JSON))
IMAGE_ID = {x["file_name"]: x["id"] for x in images}

print("GT boxes:", len(GT))


In [ ]:
# ============================================================
# Inference.
# ============================================================

BACKGROUND = {
    "__background__", "background",
    "__no_object__", "no_object", "no-object", "no object",
}

def run_inference(checkpoint, model_name):
    model = RFDETRMedium(
        pretrain_weights=str(checkpoint),
        resolution=640,
        num_classes=13,
    )

    rows = []
    times = []

    # warmup
    with Image.open(TEST_IMAGES[0]) as im:
        _ = model.predict(im.convert("RGB"), threshold=AP_CONF)

    for i, img_path in enumerate(TEST_IMAGES, 1):
        with Image.open(img_path) as im:
            image = im.convert("RGB")

        t0 = time.perf_counter()
        det = model.predict(image, threshold=AP_CONF)
        dt = time.perf_counter() - t0

        times.append({
            "model": model_name,
            "image_name": img_path.name,
            "latency_ms": dt * 1000,
        })

        if len(det):
            names = det.data["class_name"]

            for box, score, raw_name in zip(det.xyxy, det.confidence, names):
                name = (
                    raw_name.decode("utf-8")
                    if isinstance(raw_name, bytes)
                    else str(raw_name)
                ).strip().lower()

                if name in BACKGROUND:
                    continue

                assert name in NAME_TO_ID, name

                x1, y1, x2, y2 = map(float, box)
                rows.append({
                    "model": model_name,
                    "image_id": IMAGE_ID[img_path.name],
                    "image_name": img_path.name,
                    "class_id": NAME_TO_ID[name],
                    "class_name": name,
                    "score": float(score),
                    "x1": x1,
                    "y1": y1,
                    "x2": x2,
                    "y2": y2,
                })

        print(f"{model_name}: {i:02d}/20")

    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return pd.DataFrame(rows), pd.DataFrame(times)

BEFORE, BEFORE_TIME = run_inference(BEFORE_CKPT, "BEFORE_TARGET_FT")
AFTER, AFTER_TIME = run_inference(AFTER_CKPT, "AFTER_FINAL80")

BEFORE.to_csv(RESULTS / "before_target_ft_predictions.csv", index=False)
AFTER.to_csv(RESULTS / "after_final80_predictions.csv", index=False)
pd.concat([BEFORE_TIME, AFTER_TIME]).to_csv(RESULTS / "timing.csv", index=False)


In [ ]:
# ============================================================
# Evaluation helpers.
# ============================================================

def iou_xyxy(a, b):
    ax1, ay1, ax2, ay2 = map(float, a)
    bx1, by1, bx2, by2 = map(float, b)

    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)

    iw = max(0.0, ix2 - ix1)
    ih = max(0.0, iy2 - iy1)
    inter = iw * ih

    aa = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    bb = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = aa + bb - inter

    return inter / union if union > 0 else 0.0

def coco_eval(df):
    dets = [
        {
            "image_id": int(r.image_id),
            "category_id": int(r.class_id) + 1,
            "bbox": [
                float(r.x1),
                float(r.y1),
                float(r.x2 - r.x1),
                float(r.y2 - r.y1),
            ],
            "score": float(r.score),
        }
        for r in df.itertuples(index=False)
    ]

    dt = COCO_GT.loadRes(dets)
    ev = COCOeval(COCO_GT, dt, "bbox")
    ev.params.imgIds = [x["id"] for x in images]
    ev.params.catIds = list(range(1, 14))
    ev.params.maxDets = [1, 10, 100]

    with redirect_stdout(io.StringIO()):
        ev.evaluate()
        ev.accumulate()
        ev.summarize()

    summary = {
        "mAP50_95": float(ev.stats[0]),
        "mAP50": float(ev.stats[1]),
        "mAP75": float(ev.stats[2]),
    }

    precision = ev.eval["precision"]

    per_class = []
    for k, name in enumerate(CLASS_NAMES):
        arr = precision[:, :, k, 0, 2]
        arr = arr[arr > -1]

        per_class.append({
            "class_id": k,
            "class_name": name,
            "gt_boxes": int((GT["class_id"] == k).sum()),
            "AP50_95": float(arr.mean()) if arr.size else np.nan,
        })

    return summary, pd.DataFrame(per_class)

def greedy_match(gt, pred):
    pred = pred[pred["score"] >= OP_CONF].sort_values("score", ascending=False)

    used_gt = set()
    matched = []
    unmatched_pred = []

    for pi, p in pred.iterrows():
        cand = gt[
            (gt["class_id"] == p["class_id"])
            & (~gt.index.isin(used_gt))
        ]

        best_idx = None
        best_iou = -1.0

        for gi, g in cand.iterrows():
            v = iou_xyxy(
                [p.x1, p.y1, p.x2, p.y2],
                [g.x1, g.y1, g.x2, g.y2],
            )
            if v > best_iou:
                best_idx = gi
                best_iou = v

        if best_idx is not None and best_iou >= MATCH_IOU:
            used_gt.add(best_idx)
            matched.append(best_idx)
        else:
            unmatched_pred.append(pi)

    unmatched_gt = [i for i in gt.index if i not in used_gt]
    return matched, unmatched_gt, unmatched_pred

def operating_metrics(df):
    tp = fp = fn = 0
    per_image = []

    for iid in IMAGE_ID.values():
        gt = GT[GT["image_id"] == iid]
        pred = df[df["image_id"] == iid]

        m, ug, up = greedy_match(gt, pred)

        tp += len(m)
        fn += len(ug)
        fp += len(up)

        per_image.append({
            "image_id": iid,
            "TP": len(m),
            "FP": len(up),
            "FN": len(ug),
        })

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

    return {
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }, pd.DataFrame(per_image)

def size_bin(a):
    if a < 0.001:
        return "tiny"
    if a < 0.01:
        return "small"
    if a < 0.05:
        return "medium"
    return "large"

GT["size"] = GT["relative_area"].map(size_bin)

def size_recall(df):
    matched_ids = set()

    for iid in IMAGE_ID.values():
        m, _, _ = greedy_match(
            GT[GT["image_id"] == iid],
            df[df["image_id"] == iid],
        )
        matched_ids.update(m)

    rows = []
    for s in ["tiny", "small", "medium", "large"]:
        ids = set(GT[GT["size"] == s].index)
        rows.append({
            "size": s,
            "gt": len(ids),
            "matched": len(ids & matched_ids),
            "recall": len(ids & matched_ids) / len(ids) if ids else np.nan,
        })

    return pd.DataFrame(rows)


In [ ]:
# ============================================================
# Main comparison.
# ============================================================

models = {
    "BEFORE_TARGET_FT": BEFORE,
    "AFTER_FINAL80": AFTER,
}

summary_rows = []
per_class_rows = []
size_rows = []
per_image_rows = []

for name, df in models.items():
    cm, pc = coco_eval(df)
    om, pi = operating_metrics(df)
    sr = size_recall(df)

    summary_rows.append({"model": name, **cm, **om})

    pc.insert(0, "model", name)
    per_class_rows.append(pc)

    sr.insert(0, "model", name)
    size_rows.append(sr)

    pi.insert(0, "model", name)
    per_image_rows.append(pi)

SUMMARY = pd.DataFrame(summary_rows)
PER_CLASS = pd.concat(per_class_rows, ignore_index=True)
SIZE = pd.concat(size_rows, ignore_index=True)
PER_IMAGE = pd.concat(per_image_rows, ignore_index=True)

before = SUMMARY.iloc[0]
after = SUMMARY.iloc[1]

delta = {"model": "DELTA_AFTER_MINUS_BEFORE"}
for c in ["mAP50_95","mAP50","mAP75","precision","recall","f1"]:
    delta[c] = float(after[c] - before[c])
for c in ["TP","FP","FN"]:
    delta[c] = int(after[c] - before[c])

COMPARISON = pd.concat(
    [SUMMARY, pd.DataFrame([delta])],
    ignore_index=True,
)

COMPARISON.to_csv(RESULTS / "TARGET_FT_COMPARISON.csv", index=False)
PER_CLASS.to_csv(RESULTS / "PER_CLASS.csv", index=False)
SIZE.to_csv(RESULTS / "SIZE_RECALL.csv", index=False)
PER_IMAGE.to_csv(RESULTS / "PER_IMAGE.csv", index=False)

print("\n=== TARGET FT IMPACT ===")
print(COMPARISON.to_string(index=False))

print("\n=== PER CLASS ===")
print(
    PER_CLASS.pivot(
        index=["class_name","gt_boxes"],
        columns="model",
        values="AP50_95",
    ).to_string()
)

print("\n=== SIZE RECALL ===")
print(
    SIZE.pivot(
        index="size",
        columns="model",
        values="recall",
    ).to_string()
)


In [ ]:
# ============================================================
# Presentation-ready plots.
# No threshold optimization: only frozen metrics.
# ============================================================

plot_metrics = ["mAP50_95", "recall", "f1"]
plot_labels = ["mAP50-95", "Recall @0.25", "F1 @0.25"]

x = np.arange(len(plot_metrics))
width = 0.34

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width/2, [before[m] for m in plot_metrics], width, label="Before target FT")
ax.bar(x + width/2, [after[m] for m in plot_metrics], width, label="After FINAL80")
ax.set_xticks(x)
ax.set_xticklabels(plot_labels)
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("Target-domain fine-tuning impact")
ax.legend()
ax.grid(axis="y", alpha=0.25)

for i, m in enumerate(plot_metrics):
    ax.text(i-width/2, before[m]+0.02, f"{before[m]:.3f}", ha="center")
    ax.text(i+width/2, after[m]+0.02, f"{after[m]:.3f}", ha="center")

fig.tight_layout()
fig.savefig(RESULTS / "TARGET_FT_MAIN_METRICS.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:
# Per-class AP delta plot.
pivot = PER_CLASS.pivot(
    index="class_name",
    columns="model",
    values="AP50_95",
)

pivot["delta"] = pivot["AFTER_FINAL80"] - pivot["BEFORE_TARGET_FT"]
plot_df = pivot.dropna(subset=["delta"]).sort_values("delta")

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(plot_df.index, plot_df["delta"])
ax.axvline(0, linewidth=1)
ax.set_xlabel("Δ AP50-95 (AFTER - BEFORE)")
ax.set_title("Per-class impact of target-domain fine-tuning")
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
fig.savefig(RESULTS / "TARGET_FT_PER_CLASS_DELTA.png", dpi=180, bbox_inches="tight")
plt.show()


In [ ]:
# ============================================================
# 20 visual comparisons:
# GT | BEFORE TARGET FT | AFTER FINAL80
# ============================================================

FONT_PATH = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"

def get_font(size):
    if Path(FONT_PATH).exists():
        return ImageFont.truetype(FONT_PATH, size)
    return ImageFont.load_default()

def draw_label(draw, x, y, text, font, bg="black"):
    bb = draw.textbbox((x,y), text, font=font)
    pad = 2
    draw.rectangle(
        [bb[0]-pad, bb[1]-pad, bb[2]+pad, bb[3]+pad],
        fill=bg,
    )
    draw.text((x,y), text, font=font, fill="white")

def draw_gt(base, rows):
    img = base.copy()
    d = ImageDraw.Draw(img)
    f = get_font(max(12, img.width // 90))
    w = max(2, img.width // 500)

    for r in rows.itertuples(index=False):
        d.rectangle([r.x1,r.y1,r.x2,r.y2], outline="lime", width=w)
        draw_label(d, r.x1+2, max(2,r.y1+2), f"GT {r.class_name}", f, "darkgreen")

    return img

def draw_pred(base, rows):
    img = base.copy()
    d = ImageDraw.Draw(img)
    f = get_font(max(12, img.width // 90))
    w = max(2, img.width // 500)

    rows = rows[rows["score"] >= OP_CONF].sort_values("score", ascending=False)

    for r in rows.itertuples(index=False):
        d.rectangle([r.x1,r.y1,r.x2,r.y2], outline="yellow", width=w)
        draw_label(
            d,
            r.x1+2,
            max(2,r.y1+2),
            f"{r.class_name} {r.score:.2f}",
            f,
        )

    return img

def titled(img, title):
    bar_h = max(32, int(img.height * 0.045))
    out = Image.new("RGB", (img.width, img.height+bar_h), "black")
    out.paste(img, (0,bar_h))

    d = ImageDraw.Draw(out)
    f = get_font(max(16, img.width//65))
    bb = d.textbbox((0,0), title, font=f)
    tw = bb[2]-bb[0]

    d.text(((img.width-tw)/2, 5), title, font=f, fill="white")
    return out

visuals = []

for idx, img_path in enumerate(TEST_IMAGES, start=1):
    iid = IMAGE_ID[img_path.name]

    with Image.open(img_path) as im:
        base = im.convert("RGB")

    panels = [
        titled(draw_gt(base, GT[GT["image_id"]==iid]), "GROUND TRUTH"),
        titled(draw_pred(base, BEFORE[BEFORE["image_id"]==iid]), "BEFORE TARGET FT"),
        titled(draw_pred(base, AFTER[AFTER["image_id"]==iid]), "AFTER FINAL80"),
    ]

    H = max(p.height for p in panels)
    out = Image.new("RGB", (sum(p.width for p in panels), H), "black")

    x = 0
    for p in panels:
        out.paste(p, (x,0))
        x += p.width

    path = VIS / f"{idx:02d}_{img_path.stem}_TARGET_FT_COMPARE.jpg"
    out.save(path, quality=92)
    visuals.append(path)

print("Saved:", len(visuals), "visual comparisons")


In [ ]:
# ============================================================
# Contact sheet.
# ============================================================

thumb_w = 900
thumbs = []

for p in visuals:
    with Image.open(p) as im:
        img = im.convert("RGB")

    ratio = thumb_w / img.width
    img = img.resize(
        (thumb_w, int(img.height * ratio)),
        Image.Resampling.LANCZOS,
    )
    thumbs.append((p.name, img))

cols = 2
caption_h = 26
cell_h = max(x.height for _,x in thumbs) + caption_h
rows_n = math.ceil(len(thumbs)/cols)

sheet = Image.new(
    "RGB",
    (cols*thumb_w, rows_n*cell_h),
    "white",
)

d = ImageDraw.Draw(sheet)
f = get_font(16)

for i, (name, img) in enumerate(thumbs):
    r = i//cols
    c = i%cols
    x = c*thumb_w
    y = r*cell_h

    d.text((x+5,y+3), name[:90], font=f, fill="black")
    sheet.paste(img, (x,y+caption_h))

CONTACT = RESULTS / "TARGET_FT_CONTACT_SHEET.jpg"
sheet.save(CONTACT, quality=90)

print(CONTACT)


In [ ]:
# ============================================================
# Human-readable report.
# ============================================================

report = f'''# Target-domain fine-tuning impact

Comparison on the fixed 20-image TEST split.

Both models use:
- RF-DETR Medium
- 640×640 input
- full-frame inference
- confidence threshold 0.25
- IoU 0.50 for TP/FP/FN

The only major difference is target-domain adaptation:
- BEFORE: external construction_v1 only
- AFTER: fine-tuned on all 80 organizer development images

| Metric | BEFORE target FT | AFTER FINAL80 | Delta |
|---|---:|---:|---:|
| mAP50-95 | {before.mAP50_95:.6f} | {after.mAP50_95:.6f} | {after.mAP50_95-before.mAP50_95:+.6f} |
| mAP50 | {before.mAP50:.6f} | {after.mAP50:.6f} | {after.mAP50-before.mAP50:+.6f} |
| mAP75 | {before.mAP75:.6f} | {after.mAP75:.6f} | {after.mAP75-before.mAP75:+.6f} |
| Precision @0.25 | {before.precision:.6f} | {after.precision:.6f} | {after.precision-before.precision:+.6f} |
| Recall @0.25 | {before.recall:.6f} | {after.recall:.6f} | {after.recall-before.recall:+.6f} |
| F1 @0.25 | {before.f1:.6f} | {after.f1:.6f} | {after.f1-before.f1:+.6f} |
| TP | {int(before.TP)} | {int(after.TP)} | {int(after.TP-before.TP):+d} |
| FP | {int(before.FP)} | {int(after.FP)} | {int(after.FP-before.FP):+d} |
| FN | {int(before.FN)} | {int(after.FN)} | {int(after.FN-before.FN):+d} |

This comparison isolates the practical effect of target-domain fine-tuning while keeping model architecture and inference resolution fixed.

Caveat: the TEST split contains only 20 images and should be interpreted as a project-level holdout, not a large external benchmark.
'''

(RESULTS / "TARGET_FT_REPORT.md").write_text(report, encoding="utf-8")
print(report)


In [ ]:
# ============================================================
# Package outputs.
# ============================================================

archive = shutil.make_archive(
    str(WORK / "target_ft_impact_results"),
    "zip",
    root_dir=RESULTS,
)

print("\nDONE")
print("ZIP:", archive)
print("REPORT:", RESULTS / "TARGET_FT_REPORT.md")
print("MAIN PLOT:", RESULTS / "TARGET_FT_MAIN_METRICS.png")
print("CONTACT SHEET:", CONTACT)


## Output

После `Run All` скачать:

```text
/kaggle/working/target_ft_impact_results.zip
```

Внутри будут:
- `TARGET_FT_REPORT.md`
- `TARGET_FT_COMPARISON.csv`
- `PER_CLASS.csv`
- `SIZE_RECALL.csv`
- `PER_IMAGE.csv`
- `TARGET_FT_MAIN_METRICS.png`
- `TARGET_FT_PER_CLASS_DELTA.png`
- `TARGET_FT_CONTACT_SHEET.jpg`
- `visuals/` — 20 сравнений `GT | BEFORE TARGET FT | AFTER FINAL80`

Главный график и таблицу можно прямо использовать в README / презентации.
